## Reconciliation - the raw CNAF rows, mapped, carrying their codes

Clone of `clean_cnaf_1_before_qf_batch.ipynb` that drops **no column**, merged onto the
dated `*-with-codes.csv` files `generate_new_codes.ipynb` has already written.

- Replay phase 1 from the raw CNAF file, keeping every raw and intermediate column
- Load the coded rows of both CNAF routes (quotient familial, and AAH/AEEH)
- Merge them on what the export kept, giving every `id_psp` back its matricule, its
  address lines and its ORIGINESELECTION
- Write one row per code, every column, to `CNAF_RECONCILED_PATHFILE_2026`

Read-only with respect to the campaign: it writes no qf-batch input, does not touch the
phase-1 parquet and draws no code. Running it twice changes nothing.

## What differs from clean_cnaf_1_before_qf_batch.ipynb
- `cnaf.drop_raw_address_columns` is not called, so NOMCOMPLET and ADRLIG1..6 stay
- `partners.filter_rows_missing_required_fields` becomes
  `reconcile.filter_rows_missing_required_fields`: the same row filter, without the
  all-null column drop bundled into it
- the qf-batch input cell is gone, so the file qf-batch.ts reads cannot be overwritten

Row filtering is otherwise identical, so this frame holds exactly the beneficiaries phase 1
left in the parquet - which is what makes the merge below one-to-one.

## Prerequisite
`generate_new_codes.ipynb` must have run on `CNAF` and on `CNAF_AAH_AEEH`, off the very
same raw CNAF file as the one replayed here.

In [ ]:
import csv
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# partners_lib imports utils.data_utils, which lives at the data/ root: make that root
# importable first, since this notebook runs from its own directory.
try:
    import utils.data_utils  # noqa: F401
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "utils" / "data_utils.py").exists():
            sys.path.append(str(parent))
            break

# partners_lib itself sits one level up, in partners/, next to the other partner folders.
partners_root = str(Path.cwd().parent)
if partners_root not in sys.path:
    sys.path.append(partners_root)

from utils.data_utils import get_current_date_for_file_name

import partners_lib as partners
import clean_cnaf_lib as cnaf
import reconcile_cnaf_lib as reconcile

load_dotenv()

cnaf_input_filepath = os.environ['CNAF_PATHFILE_2026']

# The dated files generate_new_codes.ipynb wrote, one per CNAF route ('CNAF' and
# 'CNAF_AAH_AEEH'). Comma-separated in CNAF_WITH_CODES_PATHFILES_2026; defaults to every
# *cnaf*-with-codes.csv sitting next to DB_CNAF_EXPORT_2026, which is where
# generate_codes_lib.dated_output_path puts them.
codes_directory = Path(os.environ['DB_CNAF_EXPORT_2026']).parent

codes_filepaths_from_env = os.environ.get('CNAF_WITH_CODES_PATHFILES_2026')
codes_filepaths = (
    [Path(filepath.strip()) for filepath in codes_filepaths_from_env.split(',')]
    if codes_filepaths_from_env
    else sorted(codes_directory.glob('*cnaf*-with-codes.csv')))

# Named so it can never be picked up by the glob above on a later run.
reconciled_filepath = os.environ.get(
    'CNAF_RECONCILED_PATHFILE_2026',
    str(codes_directory / get_current_date_for_file_name('cnaf-codes-with-raw-columns.csv')))

print(f"raw CNAF: {cnaf_input_filepath}")
for codes_filepath in codes_filepaths:
    print(f"codes:    {codes_filepath}")
print(f"output:   {reconciled_filepath}")

In [ ]:
# CNAF - column names are supplied positionally, not read from the file's own header row
# (see clean_cnaf_lib.read_raw_cnaf_csv: that header row has shipped truncated before).
cnaf_df = cnaf.read_raw_cnaf_csv(cnaf_input_filepath)

print(f"{len(cnaf_df)} raw row(s) read from {cnaf_input_filepath}")

In [ ]:
# delete last row (it is not a valid row) & clean white spaces within all columns
cnaf_df = cnaf.clean_raw_cnaf(cnaf_df)

In [ ]:
# Explode postal code & commune from initial column containing both
cnaf_df = cnaf.split_postal_code_and_commune(cnaf_df)

In [ ]:
# Clean extra white spaces
cnaf_df = cnaf.normalize_full_name_spacing(cnaf_df)

In [ ]:
# map CNAF columns to the PSP schema (see cnaf.CNAF_COLUMN_MAPPING)
df_psp_mapped_cnaf = cnaf.map_cnaf_columns(cnaf_df)

In [ ]:
# Allocataire missing phone number
df_psp_mapped_cnaf = partners.clear_placeholder_phone_numbers(df_psp_mapped_cnaf)

# Allocataire's qualite
df_psp_mapped_cnaf = partners.normalize_allocataire_qualite(df_psp_mapped_cnaf)

# Additionnal address details & allocataire's street address
df_psp_mapped_cnaf = cnaf.build_allocataire_address_fields(df_psp_mapped_cnaf)

# Organism & situation - CNAF now flags the category itself, no more DOB+name guessing
df_psp_mapped_cnaf = cnaf.set_organisme_and_situation(df_psp_mapped_cnaf)

In [ ]:
# Format date_naissance to datetime python object for processing
df_psp_mapped_cnaf = partners.parse_beneficiary_birthdate(df_psp_mapped_cnaf)

## ⏸ No qf-batch input is written here
`clean_cnaf_1_before_qf_batch.ipynb` writes `QF_BATCH_INPUT_PATHFILE_2026` at this point.
That cell is deliberately absent: qf-batch.ts has already run on that file and this
notebook must not overwrite it. Nothing below needs the verdict either - it reaches these
rows through the codes files, which only exist for beneficiaries a route already selected.

In [ ]:
# clean_cnaf_1 calls cnaf.drop_raw_address_columns here, dropping NOMCOMPLET and ADRLIG1..6
# now that they have been exploded into the adresse_allocataire-* columns. They are exactly
# what this notebook exists to hand back, so they stay.
assert set(cnaf.RAW_ADDRESS_COLUMNS_TO_DROP) <= set(df_psp_mapped_cnaf.columns)

In [ ]:
# remove rows with missing necessary values (if one of those value are missing we cannot
# generate a code). Unlike phase 1 this keeps the columns left entirely null.
df_valid = reconcile.filter_rows_missing_required_fields(df_psp_mapped_cnaf)

print(f"{len(df_psp_mapped_cnaf) - len(df_valid)} row(s) removed for a missing "
      f"{partners.NECESSARY_COLUMNS}")

In [ ]:
# Upper case these columns for the merge
df_valid = partners.normalize_identity_casing(df_valid)

In [ ]:
# lower case on emails on all
df_valid = partners.normalize_email_casing(df_valid)

In [ ]:
# Preliminary filter, ahead of the precise QF/AAH/AEEH windows applied by phase 2:
# 1996-01-01 is the oldest birthdate any of the 3 routes can accept (AAH's lower bound,
# partners.AAH_DOB_MIN).
df_valid_after = partners.filter_within_eligibility_floor(df_valid)

print(f"{len(df_valid) - len(df_valid_after)} rows removed because they are outside all eligibility windows")

In [ ]:
# add missing 0 to phone numbers, and set '0' phone values to None
df_valid_after = partners.fix_phone_number_formatting(df_valid_after)

In [ ]:
# set Nan values for not existing courriel
df_valid_after = partners.clear_blank_email(df_valid_after)

In [ ]:
# add 4h on all birthdates
df_valid_after = partners.shift_birthdate_by_hours(df_valid_after)

In [ ]:
# remove duplicate beneficiaries (see partners.DEDUPLICATION_KEY_COLUMNS)
df_valid_no_duplicate, duplicate_count = partners.drop_duplicate_beneficiaries(df_valid_after)

print(f"{duplicate_count} duplicate rows were removed")

In [ ]:
# map allocataire json
df_valid_no_duplicate = partners.add_allocataire_json_column(df_valid_no_duplicate)

In [ ]:
# map adresse_allocataire json
df_valid_no_duplicate = partners.add_adresse_allocataire_json_column(df_valid_no_duplicate)

## 🔗 Reconciliation
The frame above is the phase-1 parquet plus every column phase 1 dropped. The codes files
carry the 8 columns the export kept, which are enough to key one onto the other: they are
`partners.DEDUPLICATION_KEY_COLUMNS` with the allocataire-* half folded into its JSON
column (see `reconcile.MERGE_KEY_COLUMNS`).

In [ ]:
# The key is textual on both sides: the export cast date_naissance to string right before
# writing, and the codes files are read back as text.
df_full = reconcile.format_date_naissance_as_exported(df_valid_no_duplicate)

df_full, collision_count = reconcile.drop_merge_key_collisions(df_full)
print(f"{collision_count} row(s) dropped for sharing a merge key with a row already kept")

In [ ]:
# One frame per route, tagged with the file it came from so a reconciled row says which run
# handed out its code. Read as text, the way generate_codes_lib wrote it.
df_codes = pd.concat([
    pd.read_csv(filepath, sep=';', encoding='utf-8', dtype=str, keep_default_na=False)
      .assign(fichier_codes=filepath.name)
    for filepath in codes_filepaths
], ignore_index=True)

assert df_codes['id_psp'].is_unique

print(f"{len(df_codes)} coded row(s) read from {len(codes_filepaths)} file(s)")
print(df_codes['fichier_codes'].value_counts())

In [ ]:
# Many-to-one and validated as such: the merge can neither add nor lose a code.
df_reconciled, unmatched_index = reconcile.merge_codes_with_full_rows(df_codes, df_full)

print(f"{len(df_reconciled)} reconciled row(s), {len(unmatched_index)} code(s) with no match "
      "in the raw file")

# A non-empty sample here means the raw CNAF file is not the one those codes came from.
df_reconciled.loc[unmatched_index, ['id_psp', 'nom', 'prenom', 'date_naissance',
                                    'situation', 'fichier_codes']].head(20)

In [ ]:
# The other side of the merge, for the record: phase-1 rows no code was ever handed to -
# outside their route's window, or in a household qf-batch put above the threshold.
df_without_code = reconcile.select_rows_without_code(df_full, df_codes)

print(f"{len(df_without_code)} phase-1 row(s) without a code")
print(df_without_code['situation'].value_counts(dropna=False))

In [ ]:
df_reconciled.to_csv(reconciled_filepath, sep=';', index=False, encoding='utf-8',
                     quoting=csv.QUOTE_ALL)

print(f"{len(df_reconciled)} row(s) x {len(df_reconciled.columns)} column(s) written to "
      f"{reconciled_filepath}")
print(list(df_reconciled.columns))